In [1]:
import os
import time

import geopandas as gpd
# import matplotlib.pyplot as plt
import nivapy3 as nivapy
import numpy as np
import pandas as pd
# import rasterio
# from affine import Affine
from osgeo import gdal
# from rasterio.features import geometry_mask, rasterize
# from rasterio.windows import from_bounds
from rasterstats import zonal_stats
# from shapely.ops import unary_union
# from sqlalchemy import create_engine, text
# from tqdm.notebook import tqdm

gdal.UseExceptions()

In [2]:
eng = nivapy.da.connect_postgis()

Connection successful.


# AR50 land cover proportions: rasterised workflow

In [3]:
# AR50 classes
class_dict = {
    "10": "bebygd",
    "20": "jordbruk",
    "30": "skog",
    "50": "snaumark",
    "60": "myr",
    "70": "bre",
    "81": "ferskvann",
    "82": "hav",
    "99": "ikke-kartlagt",
}

# Catchment data
cat_fpath = r"/home/jovyan/shared/common/norsink/spatial/monitored_chem_catchments/catchments_vm-rivs-quant-oc_v3-full.gpkg"
cat_id_col = "site_code"

# Raster settings
raster_res = 40  # metres
nodata = 0
tif_fpath = f"/home/jovyan/shared/common/norsink/spatial/monitored_chem_catchments/ar50_{raster_res}m.tif"
epsg = "epsg:3035"

# Output folder (result CSV added here)
catchment_set = "monitored-oc-catchs"  # for filenaming
out_folder = r"/home/jovyan/shared/common/norsink/vm_catchments_data"

In [4]:
# Read catchments
cat_gdf = gpd.read_file(cat_fpath).sort_values("area_km2", ascending=False)

cat_cea = cat_gdf.to_crs(epsg)

cat_gdf.head()

,site_code,lon,lat,site_id,Name,start_year,end_year,n_years,n_samples,av_freq,area_km2,geometry
734,002-60284,11.09243,59.270770,60284,"Glomma, Borregård B3",2011,2012,1.0,4,4.00,41698.280949,"POLYGON ((233700 6784480, 233700 6784500, 2337..."
733,002-60283,11.10059,59.268050,60283,"Glomma, Borregård B2",2011,2012,1.0,4,4.00,41696.775962,"POLYGON ((224740 6795540, 224720 6795540, 2247..."
732,002-60282,11.10680,59.263750,60282,"Glomma, Borregård B1",2011,2012,1.0,4,4.00,41696.357733,"POLYGON ((224740 6795540, 224720 6795540, 2247..."
736,002-60293,11.12306,59.270967,60293,"Glomma, Borregård",2011,2011,1.0,3,3.00,41691.863918,"POLYGON ((224740 6795540, 224720 6795540, 2247..."
477,002-30777,11.12879,59.275255,30777,Glomma ved utløp Sarpsfossen (GLOU),1986,2009,23.2,485,20.87,41691.325730,"POLYGON ((224740 6795540, 224720 6795540, 2247..."


In [5]:
# %%time

# # Does not need re-running unless something about AR50 changes.

# # Read AR50
# ar50_gdf = nivapy.da.read_postgis("physical", "norway_nibio_ar50_poly", eng)
# ar50_gdf = ar50_gdf[["artype", "geom"]]

# # Tidy
# ar50_gdf = ar50_gdf.rename(columns={"geom": "geometry"}).set_geometry("geometry")
# ar50_gdf["artype"] = ar50_gdf["artype"].astype(np.uint8)

# # Reproject
# ar50_cea = ar50_gdf.to_crs(epsg)

# ar50_cea.head()

CPU times: user 3min 22s, sys: 15.3 s, total: 3min 38s
Wall time: 3min 46s


,artype,geometry
0,50,"MULTIPOLYGON (((4081899.154 4175978.161, 40819..."
1,30,"MULTIPOLYGON (((4074149.902 4170527.681, 40741..."
2,60,"MULTIPOLYGON (((4070322.952 4163168.703, 40702..."
3,60,"MULTIPOLYGON (((4069609.138 4162293.462, 40695..."
4,60,"MULTIPOLYGON (((4074842.688 4162455.157, 40748..."


In [6]:
# %%time

# # Does not need re-running unless something about AR50 changes.

# # Rasterise
# xmin, ymin, xmax, ymax = ar50_cea.total_bounds
# width = int((xmax - xmin) / raster_res)
# height = int((ymax - ymin) / raster_res)
# transform = Affine(raster_res, 0, xmin, 0, -raster_res, ymax)
# shapes = ((geom, value) for geom, value in zip(ar50_cea.geometry, ar50_cea.artype))
# landcover_raster = rasterize(
#     shapes=shapes,
#     out_shape=(height, width),
#     transform=transform,
#     fill=nodata,
#     dtype="uint8",
# )
# with rasterio.open(
#     tif_fpath,
#     "w",
#     driver="GTiff",
#     height=height,
#     width=width,
#     count=1,
#     dtype="uint8",
#     crs=epsg,
#     transform=transform,
#     nodata=nodata,
#     compress="lzw",
# ) as dst:
#     dst.write(landcover_raster, 1)

/opt/conda/lib/python3.13/site-packages/rasterio/features.py:336: ShapeSkipWarning: Invalid or empty shape {'type': 'MultiPolygon', 'coordinates': []} at index 28057 will not be rasterized.
  warnings.warn(


CPU times: user 5min 12s, sys: 30.7 s, total: 5min 43s
Wall time: 5min 43s


In [7]:
%%time

# Zonal stats
zs = zonal_stats(cat_cea, tif_fpath, categorical=True, nodata=nodata)
stats_df = pd.DataFrame(zs).fillna(0)

# Remove nodata column if present
if nodata in stats_df.columns:
    stats_df = stats_df.drop(columns=nodata)

# Convert counts to proportions
totals = stats_df.sum(axis=1)
proportions = 100 * stats_df.div(totals.replace(0, np.nan), axis=0)

# Rename columns to class names
code_to_name = {int(k): f"{v}_pct" for k, v in class_dict.items()}
proportions = proportions.rename(columns=code_to_name)

# Merge results
cat_cea = cat_cea.reset_index(drop=True)
cat_lc_gdf = pd.concat([cat_cea, proportions], axis=1)

# Save
lu_fpath = os.path.join(out_folder, f"{catchment_set}_stats_static_lu_raster.csv")
cat_lc_gdf.drop("geometry", axis=1).to_csv(lu_fpath, index=False)

cat_lc_gdf.head()

CPU times: user 5min 34s, sys: 6.19 s, total: 5min 40s
Wall time: 5min 46s


,site_code,lon,lat,site_id,Name,start_year,end_year,n_years,n_samples,av_freq,...,geometry,bebygd_pct,jordbruk_pct,skog_pct,snaumark_pct,myr_pct,bre_pct,ferskvann_pct,hav_pct,ikke-kartlagt_pct
0,002-60284,11.09243,59.270770,60284,"Glomma, Borregård B3",2011,2012,1.0,4,4.00,...,"POLYGON ((4324076.453 4222627.717, 4324074.921...",0.688683,7.395454,49.831983,28.641486,7.989244,0.713362,4.738885,0.000834,0.00007
1,002-60283,11.10059,59.268050,60283,"Glomma, Borregård B2",2011,2012,1.0,4,4.00,...,"POLYGON ((4314269.362 4232930.525, 4314249.364...",0.687160,7.394695,49.833388,28.642187,7.989534,0.713388,4.739058,0.000520,0.00007
2,002-60282,11.10680,59.263750,60282,"Glomma, Borregård B1",2011,2012,1.0,4,4.00,...,"POLYGON ((4314269.362 4232930.525, 4314249.364...",0.686873,7.394573,49.833718,28.642359,7.989616,0.713395,4.739106,0.000291,0.00007
3,002-60293,11.12306,59.270967,60293,"Glomma, Borregård",2011,2011,1.0,3,3.00,...,"POLYGON ((4314269.362 4232930.525, 4314249.364...",0.683360,7.391399,49.836847,28.644901,7.990486,0.713473,4.739463,0.000000,0.00007
4,002-30777,11.12879,59.275255,30777,Glomma ved utløp Sarpsfossen (GLOU),1986,2009,23.2,485,20.87,...,"POLYGON ((4314269.362 4232930.525, 4314249.364...",0.682686,7.391343,49.837316,28.645128,7.990589,0.713482,4.739385,0.000000,0.00007
